# 05 · 一元线性回归推断 OLS Inference

> **能力标签**：SH6（概率与统计 / Probability & statistics）

## 目标 Objectives

完成本课后，你应该能够：

1. 推导并实现一元线性回归 OLS（Ordinary Least Squares）参数估计：$\hat{\beta}_0, \hat{\beta}_1$。
2. 计算参数的**标准误**（SE）、**t 统计量**和 **p 值**。
3. 理解 **R²**（决定系数）的含义与计算。
4. 用 `scipy.stats.linregress` 对照验证手动实现。

> 对应能力：**SH6**


## 概念讲解 Concepts

### 一元线性回归 Simple Linear Regression

模型：$y_i = \beta_0 + \beta_1 x_i + \varepsilon_i$，其中 $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$。

**OLS 估计**（最小化残差平方和）：

$$\hat{\beta} = (X^T X)^{-1} X^T y$$

其中 $X = [\mathbf{1}, \mathbf{x}]$（设计矩阵），$\mathbf{1}$ 为截距列。

---

### 推断：SE / t / p

残差 $\hat{e} = y - X\hat{\beta}$，自由度 $df = n - 2$：

$$\hat{\sigma}^2 = \frac{\hat{e}^T \hat{e}}{n - 2}$$

参数协方差矩阵：

$$\text{Cov}(\hat{\beta}) = \hat{\sigma}^2 (X^T X)^{-1}$$

标准误 $SE_j = \sqrt{\text{Cov}(\hat{\beta})_{jj}}$，t 统计量 $t_j = \hat{\beta}_j / SE_j$，双侧 $p_j = 2 P(T > |t_j|)$。

---

### 决定系数 R²

$$R^2 = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

- $R^2 \in [0, 1]$（一元 OLS）
- $R^2 = r^2$（相关系数的平方）
- $R^2 = 1$：完美拟合；$R^2 = 0$：预测等同于使用均值

---

### scipy.stats.linregress 对照

```python
from scipy import stats
slope, intercept, r, p, se = stats.linregress(x, y)
```

返回值：斜率、截距、相关系数 $r$、斜率的 p 值、斜率的标准误。


## 示例 Worked Example

In [ ]:
import numpy as np
from scipy import stats

def ols_inference(x, y) -> dict:
    """一元线性回归 y = b0 + b1*x 的推断：返回 b0,b1,se1,t1,p1（双侧）。"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    X = np.column_stack([np.ones(n), x])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    dof = n - 2
    sigma2 = (resid @ resid) / dof
    cov = sigma2 * np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    t = beta / se
    p = 2 * stats.t.sf(np.abs(t), dof)
    return {
        "beta0": float(beta[0]),
        "beta1": float(beta[1]),
        "se1":   float(se[1]),
        "t1":    float(t[1]),
        "p1":    float(p[1]),
    }

# 生成数据：y = 2 + 3x + noise
rng = np.random.default_rng(42)
n = 100
x = rng.uniform(0, 5, n)
y = 2.0 + 3.0 * x + rng.normal(0, 1.5, n)

result = ols_inference(x, y)
print("=== OLS 推断结果 ===")
print(f"  β₀ (截距) = {result['beta0']:.4f}  (真实=2.0)")
print(f"  β₁ (斜率) = {result['beta1']:.4f}  (真实=3.0)")
print(f"  SE(β₁)    = {result['se1']:.4f}")
print(f"  t(β₁)     = {result['t1']:.4f}")
print(f"  p(β₁)     = {result['p1']:.6f}")
print(f"  结论: β₁ {'显著不为零' if result['p1'] < 0.05 else '不显著'} (α=0.05)")


In [ ]:
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path

# 与 scipy.stats.linregress 对照 + R² 计算
rng = np.random.default_rng(42)
n = 100
x = rng.uniform(0, 5, n)
y = 2.0 + 3.0 * x + rng.normal(0, 1.5, n)

# scipy.stats.linregress
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
r_squared = r_value ** 2

print("=== scipy.stats.linregress ===")
print(f"  截距 = {intercept:.4f}, 斜率 = {slope:.4f}")
print(f"  SE(斜率) = {std_err:.4f}, p = {p_value:.6f}")
print(f"  R² = {r_squared:.4f}")

# 手动 R²
X = np.column_stack([np.ones(n), x])
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
y_hat = X @ beta
ss_res = np.sum((y - y_hat)**2)
ss_tot = np.sum((y - y.mean())**2)
r2_manual = 1 - ss_res / ss_tot
print(f"  手动计算 R² = {r2_manual:.4f}")
print(f"  一致: {np.isclose(r_squared, r2_manual)}")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 散点图 + 回归线
xs = np.linspace(x.min(), x.max(), 100)
axes[0].scatter(x, y, alpha=0.5, s=20, label="数据")
axes[0].plot(xs, intercept + slope * xs, "r-", lw=2, label=f"OLS: y={intercept:.2f}+{slope:.2f}x")
axes[0].set_title(f"OLS 拟合 (R²={r_squared:.3f})")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend()

# 残差图
axes[1].scatter(y_hat, y - y_hat, alpha=0.5, s=20, color="green")
axes[1].axhline(0, color="red", ls="--")
axes[1].set_title("残差图 Residual Plot")
axes[1].set_xlabel("拟合值 ŷ")
axes[1].set_ylabel("残差 y - ŷ")

fig.tight_layout()
_tmpdir = tempfile.mkdtemp()
_outpath = Path(_tmpdir) / "ols_inference_demo.png"
fig.savefig(_outpath, dpi=80)
plt.close(fig)
print("\n图像已保存到:", _outpath)


## 动手 Exercise

In [ ]:
import numpy as np
from scipy import stats

# 练习：实现完整的 ols_inference 并解读
def ols_inference(x, y) -> dict:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    X = np.column_stack([np.ones(n), x])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    dof = n - 2
    sigma2 = (resid @ resid) / dof
    cov = sigma2 * np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    t = beta / se
    p = 2 * stats.t.sf(np.abs(t), dof)
    return {"beta0": float(beta[0]), "beta1": float(beta[1]),
            "se1": float(se[1]), "t1": float(t[1]), "p1": float(p[1])}

# 场景：特征 x 与目标 y 的线性关系
rng = np.random.default_rng(77)
x_feat = rng.normal(70, 10, 80)
y_target = 4.0 + 0.05 * x_feat + rng.normal(0, 0.5, 80)

res = ols_inference(x_feat, y_target)
print("=== 特征 x → 目标 y OLS 推断 ===")
print(f"  β₀ = {res['beta0']:.4f}")
print(f"  β₁ = {res['beta1']:.4f}  (x_feat 每增 1 单位，y_target 变化 {res['beta1']:.4f})")
print(f"  SE = {res['se1']:.4f}")
print(f"  t  = {res['t1']:.4f}, p = {res['p1']:.6f}")
print(f"  结论: {'斜率显著 (p<0.05)' if res['p1'] < 0.05 else '斜率不显著'}")


## 延伸阅读 Further Reading

1. **《统计学习基础》(ESL) — Hastie et al.**：第 3 章（线性回归）（免费在线）
2. **scipy.stats.linregress 文档**：<https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html>
3. **NumPy linalg.lstsq 文档**：<https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html>
4. **StatQuest: Linear Regression**（YouTube）


## 关联练习 Related Assignments

本课对应练习包：**`w4-ols-inference`**（实现 `ols_inference()` 函数）

```bash
python check.py 04-ols-inference
```

> 能力标签：**SH6** · threshold ≥ 0.7
